# Preprocessing

## Imports

In [1]:
import cv2
import numpy as np
import os
from sklearn.model_selection import train_test_split

## Paths

In [2]:
GOOD_DIR      = r"C:\Users\wldky\OneDrive - Montana Tech\Spring-2026\CSCI 447\ML Project\dataset\cropped"
OUTPUT_DIR    = r"C:\Users\wldky\OneDrive - Montana Tech\Spring-2026\CSCI 447\ML Project\dataset\processed"

## Variables

In [3]:
IMAGE_SIZE = (224, 224)  # Resize all images to this (width, height)
VALID_GRADES = {str(i) for i in range(1, 11)}
TRAIN_RATIO = 0.70        # 70% training
VAL_RATIO = 0.15        # 15% validation
TEST_RATIO = 0.15        # 15% test
RANDOM_SEED = 42          # For reproducibility

## Functions

In [4]:
def load_and_preprocess_image(filepath):
    """
    Load a single image, resize it to IMAGE_SIZE, and normalize
    pixel values from 0-255 down to 0.0-1.0.
    Returns a flat numpy array (for use with Scikit-learn models).
    """
    img = cv2.imread(filepath)
    if img is None:
        return None

    # Resize to consistent dimensions
    img = cv2.resize(img, IMAGE_SIZE)

    # Convert BGR (OpenCV default) to RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Normalize to 0-1
    img = img.astype(np.float32) / 255.0

    # Flatten to 1D array for Scikit-learn (224 * 224 * 3 = 150,528 features)
    return img.flatten()

In [5]:
def load_dataset(good_dir=GOOD_DIR):
    X = []
    y = []

    psa_folders = sorted([
        f for f in os.listdir(good_dir)
        if os.path.isdir(os.path.join(good_dir, f)) and f.lower().startswith('psa_')
    ])

    if not psa_folders:
        print(f"No PSA_* folders found in {good_dir}")
        return None, None

    for psa_folder in psa_folders:
        grade_str = psa_folder.split('_')[1]
        if grade_str not in VALID_GRADES:
            print(f"  Skipping invalid grade folder: {psa_folder}")
            continue

        grade = int(grade_str)
        folder_path = os.path.join(good_dir, psa_folder)
        jpgs = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

        print(f"Loading {psa_folder}: {len(jpgs)} images...")

        loaded = 0
        failed = 0
        for filename in jpgs:
            filepath = os.path.join(folder_path, filename)
            img_array = load_and_preprocess_image(filepath)
            if img_array is not None:
                X.append(img_array)
                y.append(grade)
                loaded += 1
            else:
                failed += 1

        print(f"Loaded: {loaded} Failed: {failed}")

    return np.array(X), np.array(y)

In [6]:
def split_dataset(X, y):
    """
    Split X and y into train, validation, and test sets.
    Uses stratification to ensure each grade is represented
    proportionally in every split.
    """
    # First split off the test set
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y,
        test_size=TEST_RATIO,
        random_state=RANDOM_SEED,
        stratify=y  # keeps grade distribution consistent across splits
    )

    # Then split the remaining data into train and validation
    val_ratio_adjusted = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val,
        test_size=val_ratio_adjusted,
        random_state=RANDOM_SEED,
        stratify=y_train_val
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

In [7]:
def print_split_summary(y_train, y_val, y_test):
    """Print how many images of each grade ended up in each split."""
    print(f"\n{'='*50}")
    print(f"{'Grade':<10} {'Train':<10} {'Val':<10} {'Test':<10}")
    print(f"{'-'*40}")
    for grade in range(1, 11):
        train_count = np.sum(y_train == grade)
        val_count   = np.sum(y_val   == grade)
        test_count  = np.sum(y_test  == grade)
        print(f"PSA {grade:<6} {train_count:<10} {val_count:<10} {test_count:<10}")
    print(f"{'-'*40}")
    print(f"{'Total':<10} {len(y_train):<10} {len(y_val):<10} {len(y_test):<10}")
    print(f"{'='*50}")

In [8]:
def save_dataset(X_train, X_val, X_test, y_train, y_val, y_test):
    """
    Save all splits into a single .npz file.
    This is a compressed numpy format that loads very quickly during training.
    """
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    output_path = os.path.join(OUTPUT_DIR, 'dataset.npz')

    np.savez_compressed(
        output_path,
        X_train=X_train,
        X_val=X_val,
        X_test=X_test,
        y_train=y_train,
        y_val=y_val,
        y_test=y_test
    )

    size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"\nDataset saved to: {output_path}")
    print(f"File size: {size_mb:.1f} MB")

# Main Fucntion

In [9]:
print("Loading images...\n")
X, y = load_dataset()

if X is None or len(X) == 0:
    print("No images loaded. Check your GOOD_DIR path.")
else:
    print(f"\nTotal images loaded: {len(X)}")
    print(f"Image feature size:  {X.shape[1]:,} pixels per image")
    print(f"\nSplitting into train/val/test ({int(TRAIN_RATIO*100)}/{int(VAL_RATIO*100)}/{int(TEST_RATIO*100)})...")
    X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(X, y)
    print_split_summary(y_train, y_val, y_test)
    print("\nSaving dataset...")
    save_dataset(X_train, X_val, X_test, y_train, y_val, y_test)
    print("\nPreprocessing complete! Run your training script next.")

Loading images...

Loading PSA_1: 48 images...
Loaded: 48 Failed: 0
Loading PSA_10: 312 images...
Loaded: 312 Failed: 0
Loading PSA_2: 28 images...
Loaded: 28 Failed: 0
Loading PSA_3: 374 images...
Loaded: 374 Failed: 0
Loading PSA_4: 410 images...
Loaded: 410 Failed: 0
Loading PSA_5: 872 images...
Loaded: 872 Failed: 0
Loading PSA_6: 484 images...
Loaded: 484 Failed: 0
Loading PSA_7: 1758 images...
Loaded: 1758 Failed: 0
Loading PSA_8: 1894 images...
Loaded: 1894 Failed: 0
Loading PSA_9: 1452 images...
Loaded: 1452 Failed: 0

Total images loaded: 7632
Image feature size:  150,528 pixels per image

Splitting into train/val/test (70/15/15)...

Grade      Train      Val        Test      
----------------------------------------
PSA 1      34         7          7         
PSA 2      20         4          4         
PSA 3      262        56         56        
PSA 4      287        62         61        
PSA 5      610        131        131       
PSA 6      339        72         73        
